In [ ]:
conn = cx_Oracle.connect("sys", "12345", "localhost:1521/XEPDB1", mode=cx_Oracle.SYSDBA)

In [ ]:
import os
import json
import pandas as pd
import streamlit as st
from dotenv import load_dotenv
import streamlit as st
import plotly.express as px
from openai import OpenAI


from db_utils import fetch_schema, schema_to_text, run_sql, MAX_ROWS, SAFE_MODE
from llm_utils import generate_sql_json
from n8n_utils import log_event
from viz_utils import show_chart_builder, download_df

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

load_dotenv()

st.set_page_config(page_title="Oracle NL→SQL Assistant", layout="wide")
st.title("🧠 Oracle NL → SQL Assistant")

# Sidebar controls
with st.sidebar:
    st.header("Settings")
    if st.button("🔁 Refresh schema cache"):
        schema = fetch_schema(force=True)
        st.success(f"Schema refreshed: {len(schema)} tables cached.")
    safe = st.checkbox("Safe mode (SELECT/WITH only)", value=SAFE_MODE, disabled=True)
    st.caption(f"Max rows fetched: {MAX_ROWS}")

# Ensure schema cached
schema = fetch_schema(force=False)
schema_text = schema_to_text(schema, limit_tables=80, limit_cols=80)

with st.expander("📚 Schema (cached summary)", expanded=False):
    st.code(schema_text, language="text")

# Chat / query input
st.subheader("💬 Ask your database in plain English")
user_q = st.text_input("Your question", placeholder="e.g., total households by DS for MIANWALI DISTRICT, pivot by relation...")

col_run1, col_run2 = st.columns([1,1])
with col_run1:
    run_btn = st.button("✨ Generate & Run")
with col_run2:
    sql_only_btn = st.button("🧪 Generate SQL only")

if (run_btn or sql_only_btn) and not user_q.strip():
    st.warning("Please enter a question.")
    st.stop()

if run_btn or sql_only_btn:
    with st.spinner("Thinking..."):
        result = generate_sql_json(schema_text, user_q)
    st.subheader("🧾 Model decision")
    st.json(result)

    if not result.get("is_schema_query"):
        st.error(result.get("explanation") or "The question doesn't match the schema. Please rephrase to use existing tables/columns.")
        log_event("reject", {"question": user_q, "reason": result.get("explanation", "")})
        st.stop()

    sql = (result.get("sql") or "").strip()
    if not sql:
        st.error("Model didn't return SQL. Try rephrasing.")
        st.stop()

    st.subheader("🧠 Generated SQL")
    st.code(sql, language="sql")

    if sql_only_btn:
        st.info("SQL generated (not executed). Click '✨ Generate & Run' to execute.")
        st.stop()

    # Execute safely
    try:
        rows, cols = run_sql(sql)
        df = pd.DataFrame(rows, columns=cols)
        st.success(f"Query executed. Showing up to {len(df)} rows (cap: {MAX_ROWS}).")
        st.dataframe(df, use_container_width=True)

        # AI-assisted auto visualization
        st.subheader("📊 AI Visualization")
        from viz_utils import auto_plot
        auto_plot(df, user_q)


        # Download
        download_df(df)

        log_event("success", {"question": user_q, "sql": sql, "rows": len(df)})

    except Exception as e:
        st.error(f"Execution failed: {e}")
        log_event("error", {"question": user_q, "error": str(e)})



def auto_plot(df, user_question: str):
    """
    Ask LLM to suggest the best chart type + explanation for the given dataframe.
    Then plot it using Plotly.
    """
    # Schema of result
    cols = ", ".join(df.columns.tolist())
    preview = df.head(10).to_dict(orient="records")

    prompt = f"""
    You are a data visualization expert.
    User question: {user_question}
    Data columns: {cols}
    First 10 rows: {preview}

    Choose the best chart type (bar, line, scatter, pie, histogram, box).
    Respond in strict JSON:
    {{
      "chart_type": "bar|line|scatter|pie|histogram|box",
      "x": "<column for x-axis>",
      "y": "<column for y-axis or values>",
      "explanation": "one-sentence why this chart is suitable"
    }}
    """

    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "system", "content": "You are a JSON-only visualization assistant."},
                      {"role": "user", "content": prompt}],
            temperature=0
        )
        import json
        data = json.loads(resp.choices[0].message.content)
    except Exception as e:
        st.warning(f"AI failed to suggest chart: {e}")
        return

    chart_type = data.get("chart_type", "bar")
    x = data.get("x")
    y = data.get("y")
    explanation = data.get("explanation", "")

    if explanation:
        st.markdown(f"**📊 Suggested visualization:** {explanation}")

    # Build chart
    try:
        if chart_type == "bar":
            fig = px.bar(df, x=x, y=y)
        elif chart_type == "line":
            fig = px.line(df, x=x, y=y)
        elif chart_type == "scatter":
            fig = px.scatter(df, x=x, y=y)
        elif chart_type == "pie":
            fig = px.pie(df, names=x, values=y)
        elif chart_type == "histogram":
            fig = px.histogram(df, x=x)
        elif chart_type == "box":
            fig = px.box(df, x=x, y=y)
        else:
            st.warning("Unsupported chart type, falling back to bar chart.")
            fig = px.bar(df, x=x, y=y)

        st.plotly_chart(fig, use_container_width=True)

    except Exception as e:
        st.error(f"Chart rendering failed: {e}")


In [ ]:
if st.button("🔍 Show available tables"):
    non_empty_tables = []
    for table in schema.keys():
        try:
            rows, _ = run_sql(f"SELECT COUNT(*) FROM {table}")
            if rows and rows[0][0] > 0:
                non_empty_tables.append((table, rows[0][0]))
        except Exception:
            pass  # skip if table can't be queried

    if non_empty_tables:
        with st.expander("📚 Non-empty tables", expanded=True):
            for table, count in non_empty_tables:
                st.write(f"**{table}** → {count:,} rows")
    else:
        st.info("No non-empty tables found.")

In [1]:
import cx_Oracle

conn = cx_Oracle.connect("fyp", "fyp123", "localhost:1521/XEPDB1")
cur = conn.cursor()

cur.execute("SELECT user FROM dual")
print("Connected as:", cur.fetchone()[0])

cur.close()
conn.close()


Connected as: FYP


In [3]:
import cx_Oracle

conn = cx_Oracle.connect("fyp", "fyp123", "localhost:1521/XEPDB1")
cur = conn.cursor()

cur.execute("SELECT table_name FROM user_tables")
print("Tables in schema FYP:", [r[0] for r in cur.fetchall()])

cur.execute("SELECT COUNT(*) FROM POP")
print("Rows in POP:", cur.fetchone()[0])

cur.close()
conn.close()


Tables in schema FYP: ['POP']
Rows in POP: 5564560


In [ ]:
from db_utils import list_nonempty_tables

print("Connected as:", "FYP")
print("Non-empty tables:", list_nonempty_tables())



Connected as: FYP
Non-empty tables: [('POP', 5564560)]


: 

dba_utlis


In [ ]:
import os
import json
import cx_Oracle
from dotenv import load_dotenv
from typing import Dict, List, Tuple

# Load environment variables
load_dotenv()

# Configs
ORACLE_USER = os.getenv("ORACLE_USER")
ORACLE_PASSWORD = os.getenv("ORACLE_PASSWORD")
ORACLE_DSN = os.getenv("ORACLE_DSN")
ORACLE_MODE = os.getenv("ORACLE_MODE", "").upper()
SCHEMA_OWNER = os.getenv("SCHEMA_OWNER") or ORACLE_USER
MAX_ROWS = int(os.getenv("MAX_ROWS", "500"))
SAFE_MODE = os.getenv("SAFE_MODE", "true").lower() == "true"

SCHEMA_CACHE_FILE = "schema_cache.json"




def get_connection():
    """Create Oracle DB connection with optional SYSDBA mode."""
    if ORACLE_MODE == "SYSDBA":
        return cx_Oracle.connect(ORACLE_USER, ORACLE_PASSWORD, ORACLE_DSN, mode=cx_Oracle.SYSDBA)
    else:
        return cx_Oracle.connect(ORACLE_USER, ORACLE_PASSWORD, ORACLE_DSN)


def fetch_schema(force: bool = False) -> Dict[str, List[str]]:
    """Read schema (tables + columns) for SCHEMA_OWNER and cache to schema_cache.json."""
    if (not force) and os.path.exists(SCHEMA_CACHE_FILE):
        try:
            with open(SCHEMA_CACHE_FILE, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass  # Ignore and rebuild

    conn = get_connection()
    cur = conn.cursor()
    cur.execute("""
        SELECT table_name, column_name
        FROM all_tab_columns
        WHERE owner = UPPER(:owner)
        ORDER BY table_name, column_id
    """, owner=SCHEMA_OWNER)

    schema = {}
    for table_name, column_name in cur.fetchall():
        t = table_name.strip().upper()
        c = column_name.strip().upper()
        schema.setdefault(t, []).append(c)

    cur.close()
    conn.close()

    with open(SCHEMA_CACHE_FILE, "w", encoding="utf-8") as f:
        json.dump(schema, f, indent=2)

    return schema


def schema_to_text(schema: Dict[str, List[str]], limit_tables: int = 40, limit_cols: int = 60) -> str:
    """Render a compact text representation of the schema for LLM prompts."""
    lines = []
    for i, (t, cols) in enumerate(schema.items()):
        if i >= limit_tables: 
            lines.append(f"... ({len(schema) - limit_tables} more tables omitted)")
            break
        shown = cols[:limit_cols]
        extra = "" if len(cols) <= limit_cols else f", ... (+{len(cols)-limit_cols})"
        lines.append(f"{t}({', '.join(shown)}{extra})")
    return "\n".join(lines)


def is_safe_select(sql: str) -> bool:
    """In SAFE_MODE, allow only SELECT/WITH statements."""
    s = (sql or "").lstrip().upper()
    return s.startswith("SELECT") or s.startswith("WITH")


def run_sql(sql: str) -> Tuple[list, list]:
    """Execute SQL (read-only by default). Returns (rows, column_names)."""
    if SAFE_MODE and not is_safe_select(sql):
        raise RuntimeError("Unsafe SQL blocked. Only SELECT/WITH allowed (SAFE_MODE=true).")

    conn = get_connection()
    cur = conn.cursor()
    cur.execute(sql)

    colnames = [d[0] for d in cur.description] if cur.description else []
    rows = []
    remaining = MAX_ROWS

    while remaining > 0:
        chunk = cur.fetchmany(min(remaining, 1000))
        if not chunk:
            break
        rows.extend(chunk)
        remaining -= len(chunk)

    cur.close()
    conn.close()
    return rows, colnames
def list_nonempty_tables() -> list:
    """
    Return list of non-empty tables in SCHEMA_OWNER with their row count and columns.
    Format: [(table_name, row_count, [col1, col2, ...]), ...]
    """
    conn = get_connection()
    cur = conn.cursor()

    # Get all tables owned by schema
    cur.execute("""
        SELECT table_name
        FROM all_tables
        WHERE owner = UPPER(:owner)
    """, owner=SCHEMA_OWNER)

    tables = [r[0] for r in cur.fetchall()]
    nonempty = []

    for t in tables:
        try:
            # Count rows
            cur.execute(f'SELECT COUNT(*) FROM "{t}"')
            count = cur.fetchone()[0]

            if count > 0:
                # Fetch columns
                cur.execute("""
                    SELECT column_name
                    FROM all_tab_columns
                    WHERE owner = UPPER(:owner) AND table_name = :t
                    ORDER BY column_id
                """, owner=SCHEMA_OWNER, t=t)

                cols = [r[0] for r in cur.fetchall()]
                nonempty.append((t, count, cols))

        except Exception as e:
            print(f"⚠️ Could not fetch info for {t}: {e}")

    cur.close()
    conn.close()
    return nonempty


In [ ]:
# ✅ Show table only for main query
                if idx == 1:
                    st.markdown(f"### 📊 Visualizations for Main Query ({item.get('explanation','')})")
                    #st.dataframe(df.head(10), use_container_width=True)
                else:
                    st.markdown(f"### 📊 Extra Visualization {idx-1}: {item.get('explanation','')}")

                 # 🔥 Chart suggestions
                def auto_plot(df, user_question: str, idx: int):
                    cols = ", ".join(df.columns.tolist())
                    preview = df.head(10).to_dict(orient="records")

                    prompt = f"""
                    You are a visualization expert.
                    User question: {user_question}
                    Data columns: {cols}
                    First 10 rows: {preview}

                    Suggest 1 good chart (bar, line, scatter, pie, histogram, box).
                    Respond JSON:
                    {{
                      "chart_type": "bar|line|scatter|pie|histogram|box",
                      "x": "<col>",
                      "y": "<col>",
                      "explanation": "why this chart is useful"
                    }}
                    """
                    try:
                        resp = client.chat.completions.create(
                            model="gpt-4o-mini",
                            messages=[
                                {"role": "system", "content": "You are a JSON-only visualization assistant."},
                                {"role": "user", "content": prompt}
                            ],
                            temperature=0
                        )
                        chart = json.loads(resp.choices[0].message.content)
                    except Exception as e:
                        st.warning(f"Chart suggestion failed: {e}")
                        return

                    chart_type = chart.get("chart_type", "bar")
                    x, y = chart.get("x"), chart.get("y")
                    explanation = chart.get("explanation", "")

                    if explanation:
                        st.markdown(f"**{explanation}**")

                    try:
                        if chart_type == "bar":
                            fig = px.bar(df, x=x, y=y, color=x)
                        elif chart_type == "line":
                            fig = px.line(df, x=x, y=y, markers=True)
                        elif chart_type == "scatter":
                            fig = px.scatter(df, x=x, y=y, size=y, color=x)
                        elif chart_type == "pie":
                            values = y if isinstance(y, str) else y[0]
                            fig = px.pie(df, names=x, values=values, hole=0.3)
                        elif chart_type == "histogram":
                            fig = px.histogram(df, x=x)
                        elif chart_type == "box":
                            fig = px.box(df, x=x, y=y, points="all")
                        else:
                            fig = px.bar(df, x=x, y=y)

                        # ✅ Black labels + distinct style
                        fig.update_layout(
                            template="plotly_white",
                            title=f"{chart_type.title()} Chart {idx}",
                            font=dict(color="black"),
                            legend=dict(font=dict(color="black"))
                        )
                        st.plotly_chart(fig, use_container_width=True)
                    except Exception as e:
                        st.error(f"Chart rendering failed: {e}")

                auto_plot(df, user_q, idx)

In [ ]:
# --- Main execution ---
if run_btn or sql_only_btn:
    with st.spinner("Thinking..."):
        result = generate_sql_json(schema_text, user_q)

    st.subheader("🧾 Model decision")
    st.json(result)

    if not result.get("is_schema_query"):
        st.error(result.get("explanation") or "The question doesn't match the schema. Please rephrase to use existing tables/columns.")
        log_event("reject", {"question": user_q, "reason": result.get("explanation", "")})
        st.stop()

    sql = (result.get("sql") or "").strip()
    if not sql:
        st.error("Model didn't return SQL. Try rephrasing.")
        st.stop()

    # 🚨 Validate SQL
    if not validate_sql(sql, schema):
        st.error("❌ Invalid query. You can only query the available tables: " + ", ".join(schema.keys()))
        st.stop()

    st.subheader("🧠 Generated SQL")
    st.code(sql, language="sql")

    if sql_only_btn:
        st.info("SQL generated (not executed). Click '✨ Generate & Run' to execute.")
        st.stop()

    # Execute safely
    try:
        rows, cols = run_sql(sql)
        df = pd.DataFrame(rows, columns=cols)
        st.success(f"✅ Query executed. Showing up to {len(df)} rows (cap: {MAX_ROWS}).")
        st.dataframe(df, use_container_width=True)

        # --- AI-assisted auto visualization ---
        st.subheader("📊 AI Visualization")
        def auto_plot(df, user_question: str):
            """
            Ask LLM to suggest the best chart type + explanation for the given dataframe.
            Then plot it using Plotly.
            """
            cols = ", ".join(df.columns.tolist())
            preview = df.head(10).to_dict(orient="records")

            prompt = f"""
            You are a data visualization expert.
            User question: {user_question}
            Data columns: {cols}
            First 10 rows: {preview}

            Choose the best chart type (bar, line, scatter, pie, histogram, box).
            Respond in strict JSON:
            {{
              "chart_type": "bar|line|scatter|pie|histogram|box",
              "x": "<column for x-axis>",
              "y": "<column for y-axis or values>",
              "explanation": "one-sentence why this chart is suitable"
            }}
            """

            try:
                resp = client.chat.completions.create(
                    model="gpt-4o-mini",
                    messages=[
                        {"role": "system", "content": "You are a JSON-only visualization assistant."},
                        {"role": "user", "content": prompt}
                    ],
                    temperature=0
                )
                data = json.loads(resp.choices[0].message.content)
            except Exception as e:
                st.warning(f"AI failed to suggest chart: {e}")
                return

            chart_type = data.get("chart_type", "bar")
            x = data.get("x")
            y = data.get("y")
            explanation = data.get("explanation", "")

            # --- Normalize x and y ---
            if isinstance(x, str) and "," in x:
                x = [col.strip() for col in x.split(",")]
            if isinstance(y, str) and "," in y:
                y = [col.strip() for col in y.split(",")]

            if explanation:
                st.markdown(f"**📊 Suggested visualization:** {explanation}")

            try:
                if chart_type == "bar":
                    fig = px.bar(df, x=x, y=y)
                elif chart_type == "line":
                    fig = px.line(df, x=x, y=y)
                elif chart_type == "scatter":
                    fig = px.scatter(df, x=x, y=y)
                elif chart_type == "pie":
                    # Pie only supports single value column
                    values = y if isinstance(y, str) else y[0]
                    fig = px.pie(df, names=x, values=values)
                elif chart_type == "histogram":
                    fig = px.histogram(df, x=x)
                elif chart_type == "box":
                    fig = px.box(df, x=x, y=y)
                else:
                    st.warning("Unsupported chart type, falling back to bar chart.")
                    fig = px.bar(df, x=x, y=y)

                st.plotly_chart(fig, use_container_width=True)

            except Exception as e:
                st.error(f"Chart rendering failed: {e}")

        auto_plot(df, user_q)

        # --- Download ---
        download_df(df)

        log_event("success", {"question": user_q, "sql": sql, "rows": len(df)})

    except Exception as e:
        st.error(f"Execution failed: {e}")
        log_event("error", {"question": user_q, "error": str(e)})



In [ ]:
# 🎨 Universal chart renderer
def render_chart(df: pd.DataFrame, chart_type: str, title: str):
    fig = None
    cols = df.columns.tolist()

    try:
        if chart_type == "bar":
            fig = px.bar(df, x=cols[0], y=cols[1], title=title, text_auto=True)

        elif chart_type == "stacked_bar":
            if len(cols) >= 3:
                fig = px.bar(df, x=cols[0], y=cols[1], color=cols[2], title=title, text_auto=True)
            else:
                fig = px.bar(df, x=cols[0], y=cols[1], title=title, text_auto=True)

        elif chart_type == "grouped_bar":
            if len(cols) >= 3:
                fig = px.bar(df, x=cols[0], y=cols[1], color=cols[2], barmode="group", title=title)
            else:
                fig = px.bar(df, x=cols[0], y=cols[1], title=title)

        elif chart_type == "pie":
            fig = px.pie(df, names=cols[0], values=cols[1], title=title)

        elif chart_type == "donut":
            fig = px.pie(df, names=cols[0], values=cols[1], hole=0.4, title=title)

        elif chart_type == "line":
            fig = px.line(df, x=cols[0], y=cols[1], title=title, markers=True)

        elif chart_type == "area":
            fig = px.area(df, x=cols[0], y=cols[1], title=title)

        elif chart_type == "scatter":
            fig = px.scatter(df, x=cols[0], y=cols[1], size=df[cols[1]], color=cols[0], title=title)

        elif chart_type == "bubble":
            if len(cols) >= 3:
                fig = px.scatter(df, x=cols[0], y=cols[1], size=cols[2], color=cols[0], title=title)
            else:
                fig = px.scatter(df, x=cols[0], y=cols[1], size=df[cols[1]], color=cols[0], title=title)

        elif chart_type == "histogram":
            fig = px.histogram(df, x=cols[0], title=title)

        elif chart_type == "box":
            fig = px.box(df, y=cols[0], title=title)

        elif chart_type == "heatmap":
            if len(cols) >= 3:
                fig = px.density_heatmap(df, x=cols[0], y=cols[1], z=cols[2], title=title, text_auto=True)
            else:
                fig = px.density_heatmap(df, x=cols[0], y=cols[1], title=title, text_auto=True)

        elif chart_type == "treemap":
            if len(cols) >= 3:
                fig = px.treemap(df, path=[cols[0], cols[1]], values=cols[2], title=title)
            else:
                fig = px.treemap(df, path=[cols[0]], values=cols[1], title=title)

        elif chart_type == "sunburst":
            if len(cols) >= 3:
                fig = px.sunburst(df, path=[cols[0], cols[1]], values=cols[2], title=title)
            else:
                fig = px.sunburst(df, path=[cols[0]], values=cols[1], title=title)

        elif chart_type == "funnel":
            fig = px.funnel(df, x=cols[0], y=cols[1], title=title)

        elif chart_type == "radar":
            fig = go.Figure()
            fig.add_trace(go.Scatterpolar(r=df[cols[1]], theta=df[cols[0]], fill="toself", name=title))
            fig.update_layout(polar=dict(radialaxis=dict(visible=True)), title=title)

        elif chart_type == "polar":
            fig = px.bar_polar(df, r=cols[1], theta=cols[0], color=cols[0], title=title)

        elif chart_type == "gauge":
            # ✅ Gauge chart (single metric)
            value = df[cols[1]].iloc[0] if len(df) > 0 else 0
            fig = go.Figure(go.Indicator(
                mode="gauge+number",
                value=value,
                title={'text': title},
                gauge={'axis': {'range': [None, value * 1.5]}}
            ))

        else:
            st.warning(f"⚠️ Unsupported chart type: {chart_type}")
            return

        st.plotly_chart(fig, use_container_width=True)

    except Exception as e:
        st.error(f"Chart rendering failed: {e}")


# 🎯 Main Flow
st.title("📊 AI-Powered Multi-Query Storyboards")

user_question = st.text_input("Ask your question (e.g., show household per districts):")

if user_question:
    schema_text = fetch_schema()

    with st.spinner("🤖 Generating queries and charts..."):
        result = generate_sql_json(schema_text, user_question)

    if not result.get("is_schema_query", False):
        st.error(result.get("reason", "Query could not be generated."))
    else:
        for item in result.get("items", []):
            sql = item["sql"]
            chart = item["chart"]
            title = item.get("title", "Chart")
            explanation = item.get("explanation", "")

            try:
                rows, cols, final_sql = execute_with_autofix(sql, user_question, schema_text)
                df = pd.DataFrame(rows, columns=cols)

                if df.empty:
                    st.warning(f"No data for query: {title}")
                else:
                    st.markdown(f"### {title}")
                    if explanation:
                        st.caption(explanation)

                    render_chart(df, chart, title)

            except Exception as e:
                st.error(f"❌ SQL Execution failed: {e}")
